# Sandbox

A scratch pad for looking at every character the **US International Scientific**
keyboard layout can produce.

The layout is read straight from `layout/us-intl-scientific.toml` through the
shared loader in `tools/kbdlayout`, the same one the generators and the checks
use, so nothing here can drift away from what the layout actually says.

In [ ]:
import random
import sys
from collections import Counter
from pathlib import Path

REPO = Path.cwd()
while not (REPO / 'layout/us-intl-scientific.toml').exists():
    if REPO.parent == REPO:
        raise SystemExit('run this notebook from inside the repository')
    REPO = REPO.parent

sys.path.insert(0, str(REPO / 'tools'))
from kbdlayout import source
from kbdlayout.model import LEVEL_NAMES, LEVELS

layout = source.load(REPO / 'layout/us-intl-scientific.toml')
print(f'{layout.name} {layout.version}')
print(f'{len(layout.keys)} keys, {len(layout.dead_keys)} dead keys')

## Characters on the keys themselves

`LEVELS` are the five shift states a key can carry: unmodified, Shift, Ctrl,
AltGr and AltGr + Shift.

In [ ]:
for level in LEVELS:
    characters = ''.join(
        key.outputs[level].char for key in layout.keys if level in key.outputs
    )
    print(f'{LEVEL_NAMES[level]:>14}: {characters}')

## Characters behind the dead keys

In [ ]:
for dead_key in layout.dead_keys:
    composites = ''.join(chr(composite) for _base, composite in dead_key.entries)
    print(f'U+{dead_key.root:04X} {dead_key.root_name:<38} {composites}')

## The whole inventory

Every character the layout can produce, counted once per code point. A code
point that shows up more than once is reachable in more than one way -- that is
normal: the ISO extra key repeats the backslash key, and several dead keys share
a default character.

In [ ]:
everything = Counter()
for key in layout.keys:
    for output in key.outputs.values():
        everything[output.code_point] += 1
for dead_key in layout.dead_keys:
    for _base, composite in dead_key.entries:
        everything[composite] += 1

print(f'{sum(everything.values())} mappings, {len(everything)} distinct characters')
print(f'highest code point: U+{max(everything):04X}')
print('reachable more than once:',
      ''.join(chr(cp) for cp, n in sorted(everything.items()) if n > 1))

In [ ]:
characters = sorted(everything)
random.shuffle(characters)
width = 30
for start in range(0, len(characters), width):
    print(''.join(chr(cp) for cp in characters[start:start + width]))

## Look up a single character

In [ ]:
def where(character):
    """Show every way `character` can be typed."""
    code_point = ord(character)
    for key in layout.keys:
        for level, output in key.outputs.items():
            if output.code_point == code_point:
                print(f'key {key.id} in the {LEVEL_NAMES[level]} shift state')
    for dead_key in layout.dead_keys:
        for base, composite in dead_key.entries:
            if composite == code_point:
                print(f'dead key U+{dead_key.root:04X} then {chr(base)!r}')


where('ǎ')

## What each platform gets

The same layout is written out for Windows and for Linux by
`python3 tools/generate.py`; this is what those files contain.

In [ ]:
from kbdlayout.checks_generated import render_all

for path, content in sorted(render_all(layout).items()):
    size = len(content if isinstance(content, bytes) else content.encode('utf-8'))
    print(f'{size:>7} bytes  {path}')